# 01 Data Audit

This notebook performs the first data audit for the customer segmentation project. It loads the raw datasets, checks structure and data quality, validates customer ID overlap, parses basket product lists, and reports basic basket statistics.

No clustering or final output generation is performed in this notebook.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)


def find_project_root(start: Path) -> Path:
    """Find the repository root from the current notebook location."""
    for candidate in [start, *start.parents]:
        if (candidate / "customer_info.csv").exists() and (candidate / "customer_basket.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate customer_info.csv and customer_basket.csv from the current path.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.data_loading import load_datasets
from src.data_audit import (
    basket_length_distribution,
    dataframe_overview,
    duplicate_summary,
    flag_suspicious_ranges,
    missing_values,
    parse_goods_column,
    top_products,
    validate_customer_overlap,
)

## Load Data

In [ ]:
customer_info, customer_basket = load_datasets(PROJECT_ROOT)

print(f"customer_info shape: {customer_info.shape}")
print(f"customer_basket shape: {customer_basket.shape}")

## Columns and Data Types

In [ ]:
print("customer_info columns:")
print(list(customer_info.columns))
display(dataframe_overview(customer_info))

print("customer_basket columns:")
print(list(customer_basket.columns))
display(dataframe_overview(customer_basket))

## Missing Values

In [ ]:
customer_info_missing = missing_values(customer_info)
customer_basket_missing = missing_values(customer_basket)

print("customer_info missing values:")
display(customer_info_missing if not customer_info_missing.empty else pd.DataFrame({"message": ["No missing values found."]}))

print("customer_basket missing values:")
display(customer_basket_missing if not customer_basket_missing.empty else pd.DataFrame({"message": ["No missing values found."]}))

## Duplicate Checks

In [ ]:
duplicates = duplicate_summary(customer_info, customer_basket)
display(duplicates)

## Customer ID Overlap

In [ ]:
overlap = validate_customer_overlap(customer_info, customer_basket)
display(pd.DataFrame([overlap]))

assert overlap["basket_ids_missing_in_info"] == 0, "Some basket customer IDs are missing from customer_info."

print("PASS: every customer_id in customer_basket exists in customer_info.")
print(
    "Customers in customer_info without sampled baskets: "
    f"{overlap['customers_without_baskets']} ({overlap['customers_without_baskets_pct']}%)"
)

## Parse Basket Goods

In [ ]:
parsed_basket, parse_errors = parse_goods_column(customer_basket)

print(f"Rows parsed: {len(parsed_basket):,}")
print(f"Parse errors: {len(parse_errors):,}")
display(parse_errors.head() if not parse_errors.empty else pd.DataFrame({"message": ["No parse errors found."]}))

assert parse_errors.empty, "Some list_of_goods values could not be parsed."

display(parsed_basket[["invoice_id", "customer_id", "goods", "basket_length"]].head())

## Basket Length Distribution

In [ ]:
length_distribution = basket_length_distribution(parsed_basket)
display(length_distribution)

## Top Products

In [ ]:
top_product_table = top_products(parsed_basket, top_n=20)
display(top_product_table)

## Suspicious Range Checks

In [ ]:
range_flags = flag_suspicious_ranges(customer_info, parsed_basket)
display(range_flags if not range_flags.empty else pd.DataFrame({"message": ["No suspicious ranges found."]}))

## Audit Summary

In [ ]:
audit_summary = {
    "customer_info_rows": len(customer_info),
    "customer_info_columns": customer_info.shape[1],
    "customer_basket_rows": len(customer_basket),
    "customer_basket_columns": customer_basket.shape[1],
    "basket_ids_missing_in_info": overlap["basket_ids_missing_in_info"],
    "customers_without_baskets": overlap["customers_without_baskets"],
    "parse_errors": len(parse_errors),
    "suspicious_range_flags": len(range_flags),
}

display(pd.DataFrame([audit_summary]))
print("Audit complete. No clustering was performed.")